# Baseline Experiment - Binary Classification - BERT

Apply BERT to classify whether a tweet is relevant to a disater or not, in other word, binary classification.

## A. Setup

In [ ]:
%pip install transformers torch tqdm datasets

In [ ]:
from pathlib import Path
# import os

# from dotenv import load_dotenv
# load_dotenv()

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification

import configuration
from src import setup, data_utils, hf_utils
from src.models import bert

In [ ]:
data_frac = data_utils.DATA_FRACTION

# https://huggingface.co/google-bert/bert-base-uncased
# bert-base-uncased - 110M
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

device = setup.setup_device_with_seeds()

batch_size = 32
learning_rate = 1e-5
num_epochs = 10
patience = 3  # early stopping, if validation loss does not improve for this many epochs

bert_base = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2  # e.g., binary sentiment
)
bert_base.config.problem_type = "single_label_classification"
# Optimizer
optimizer = AdamW(bert_base.parameters(), lr=learning_rate)

base_configs = {
    "bert": bert_base,
    "batch_size": batch_size,
    "optimizer": optimizer,
    "num_epochs": num_epochs,
    "patience": patience,
    "device": device,
}

In [ ]:
def run_experiment(weather_ratio, out_topic_ratio):
    
    df_train, df_val, df_test = data_utils.load_BERT_sets(
        weather_ratio=weather_ratio, out_topic_ratio=out_topic_ratio
    )
    
    # ====
    # Comment out this cell to use the full dataset. This is just for quick testing.
    train_size = 1000
    val_size = int(train_size * len(df_val) / len(df_train))
    test_size = int(train_size * len(df_test) / len(df_train))

    df_train = df_train.sample(n=train_size, random_state=setup.RANDOM_SEED)
    df_val = df_val.sample(n=val_size, random_state=setup.RANDOM_SEED)
    df_test = df_test.sample(n=test_size, random_state=setup.RANDOM_SEED)
    # ----

    ds_train, ds_val, ds_test = hf_utils.create_datasets(df_train, df_val, df_test)

    # Calculate the maximum length of the tokenized tweets in the training set to set the max_length parameter for BERT
    hf_utils.max_length_dist(df_train, "tweet_text", tokenizer)
    
    ration_folder = data_utils.get_experiment_ratios_path(weather_ratio, out_topic_ratio)

    # Tokenize the datasets and save the tokenized versions.
    token_path = Path(
        f"../tokens/BERT/{ration_folder}"
    )
    train_tokenized, val_tokenized, test_tokenized = hf_utils.load_or_tokenize(
        ds_train,
        ds_val,
        ds_test,
        tokenizer,
        token_path,
        force_retokenize=True,
        format_dataset=bert.format_dataset,
    )
    
    imbalance_strategy = bert.detect_imbalance_strategy(df_train["informative"])
    print(f"Detected imbalance strategy: {imbalance_strategy}")
    
    configs = {**base_configs, **imbalance_strategy}
    configs['save_path'] = f"../models/BERT/{ration_folder}"

    # Fine-tune BERT
    model, train_loss_history, val_loss_history, val_f1_history, val_recall_history, val_precision_history = bert.finetune(
        train_tokenized, val_tokenized, configs
    )

    # Visualize training history
    hf_utils.plot_fine_tune_history(
        train_loss_history, val_loss_history, val_f1_history, val_recall_history, val_precision_history
    )

    predictions = bert.predict(model, test_tokenized, device)
    bert.report_metrics(test_tokenized, predictions)
    hf_utils.group_report_metrics(
        df_test, predictions, group_by="subset", labels="informative"
    )

In [ ]:
for weather_ratio, out_topic_ratio in data_utils.BERT_EXPERIMENT_RATIOS:
    print(
        f"Processing dataset with weather_ratio={weather_ratio} and out_topic_ratio={out_topic_ratio}..."
    )
    run_experiment(weather_ratio, out_topic_ratio)